# P2-3 · CNN hiện đại và transfer learning

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torchvision
num_classes = 5
WEIGHTS = None     # thực tế: "IMAGENET1K_V1" (tải ~45 MB trọng số pretrain); trang này không tải được nên dùng None để chạy thử
m = torchvision.models.resnet18(weights=WEIGHTS)
m.fc = torch.nn.Linear(m.fc.in_features, num_classes)        # head mới: 512 → num_classes
count = lambda: sum(p.numel() for p in m.parameters() if p.requires_grad)

# Giai đoạn 1: linear probe
for p in m.parameters(): p.requires_grad = False
for p in m.fc.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(m.fc.parameters(), lr=1e-3)
print("giai đoạn 1, tham số được train:", count())          # chỉ head: 512 × 5 + 5
# ... train vài epoch ...

# Giai đoạn 2: fine-tune toàn bộ, backbone lr nhỏ hơn
for p in m.parameters(): p.requires_grad = True
opt = torch.optim.AdamW([
    {"params": m.fc.parameters(), "lr": 1e-3},
    {"params": [p for n, p in m.named_parameters() if not n.startswith("fc")], "lr": 1e-4},
], weight_decay=0.01)
print("giai đoạn 2, tham số được train:", count())

giai đoạn 1, tham số được train: 2565
giai đoạn 2, tham số được train: 11179077


In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()      # resize, crop, chuẩn hóa đúng như lúc pretrain
preprocess

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
import torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"                            # fp16 chỉ có lợi trên GPU
model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_loader = [(torch.randn(64, 3, 32, 32), torch.randint(0, 10, (64,))) for _ in range(5)]   # dữ liệu giả

scaler = torch.amp.GradScaler(device, enabled=use_amp)      # chỉ cần cho fp16
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    opt.zero_grad()
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        loss = F.cross_entropy(model(x), y)
    scaler.scale(loss).backward()                     # nhân loss lên để gradient nhỏ không thành 0
    scaler.step(opt)                                  # chia lại trước khi cập nhật, bỏ bước nếu có inf
    scaler.update()
print("thiết bị:", device, "| fp16:", use_amp, "| loss cuối:", round(loss.item(), 3))

thiết bị: cpu | fp16: False | loss cuối: 2.385


## Bài tập 5: Debug một vòng train hỏng {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
X = torch.randn(1024, 20); Y = (X @ torch.randn(20, 10)).argmax(1)       # 10 lớp, có quy luật tuyến tính
loader = [(X[i:i + 64], Y[i:i + 64]) for i in range(0, 1024, 64)]
criterion = nn.CrossEntropyLoss()
def make():
    torch.manual_seed(1)
    m = nn.Sequential(nn.Linear(20, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 10))
    return m, torch.optim.SGD(m.parameters(), lr=0.05)

In [ ]:
model, optimizer = make()
for epoch in range(5):
    model.eval()
    for x, y in loader:
        logits = model(x)
        loss = criterion(torch.softmax(logits, dim=1), y)
        loss.backward()
        optimizer.step()
    print("epoch", epoch, "loss", round(loss.item(), 3))

epoch 0 loss 2.299
epoch 1 loss 2.265
epoch 2 loss 2.22
epoch 3 loss 2.213
epoch 4 loss 2.132


In [ ]:
model, optimizer = make()
for epoch in range(5):
    model.train()
    for x, y in loader:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    print("epoch", epoch, "loss", round(loss.item(), 3))

epoch 0 loss 2.21
epoch 1 loss 2.003
epoch 2 loss 1.849
epoch 3 loss 1.723
epoch 4 loss 1.58
